## Previous floodfill implementation

In [ ]:
def get_neighborhood(z, y, x, shape):
    z, y, x = int(round(z)), int(round(y)), int(round(x))
    neighbors = []
    for i in range(-1, 2):
        for j in range(-1, 2):
            for k in range(-1, 2):
                if i + j + k == 0:
                    continue
                nz, ny, nx = z + k, y + j, x + i
                if 0 <= nz < shape[0] and 0 <= ny < shape[1] and 0 <= nx < shape[2]:
                    neighbors.append((nz, ny, nx))
    return neighbors

In [ ]:

from collections import deque
import numpy as np

def floodfill_opt(im, initial_mask, forbidden_mask=None):
    """
    Flood-fills a spine region across Z slices from sparse mask annotations,
    using intensity and 3D connectivity. Optionally excludes 'forbidden' voxels.

    Parameters:
    - im: 3D image stack (Z, Y, X)
    - initial_mask: 3D binary mask for current spine
    - forbidden_mask: 3D binary mask where fill is not allowed (e.g., dendrites)

    Returns:
    - final_mask: 3D binary mask after flood fill
    """
    final_mask = np.zeros_like(im, dtype=bool)
    final_mask[initial_mask] = True

    alpha = 0.8

    # Use a high-intensity voxel from the mask as seed
    zyx_coords = np.argwhere(initial_mask)
    seed_z, seed_y, seed_x = zyx_coords[np.argmax(im[initial_mask])]
    seed_intensity = float(im[seed_z, seed_y, seed_x])
    threshold = min(seed_intensity * alpha, np.percentile(im[initial_mask], 85))

    # Initialize queue
    queue = deque()
    queue.extend(get_neighborhood(seed_x, seed_y, seed_z, im.shape))

    while queue:
        x, y, z = queue.popleft()

        # Bounds check
        if not (0 <= z < im.shape[0] and 0 <= y < im.shape[1] and 0 <= x < im.shape[2]):
            continue

        # Already visited or forbidden (like dendrite)
        if final_mask[z, y, x]:
            continue
        if forbidden_mask is not None and forbidden_mask[z, y, x]:
            continue

        # Intensity check
        if im[z, y, x] > threshold:
            final_mask[z, y, x] = True
            neighbors = get_neighborhood(x, y, z, im.shape)
            for nx, ny, nz in neighbors:
                if forbidden_mask is not None and forbidden_mask[nz, ny, nx]:
                    continue
                if final_mask[nz, ny, nx]:
                    continue
                queue.append((nx, ny, nz))

    return final_mask

## Utilities for plotting

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

def plot_masked_spines_3d(img, initial_mask, flooded_mask):
    zs, ys, xs = np.where(flooded_mask)
    pad = 100
    zmin, zmax = max(0, zs.min()-1), min(zs.max()+1, flooded_mask.shape[0])
    ymin, ymax = max(0, ys.min()-pad), min(ys.max()+pad, flooded_mask.shape[1])
    xmin, xmax = max(0, xs.min()-pad), min(xs.max()+pad, flooded_mask.shape[2])

    # delimit spine section
    img_spine = img[zmin:zmax, ymin:ymax, xmin:xmax]
    initial_mask_spine = initial_mask[zmin:zmax, ymin:ymax, xmin:xmax]
    flooded_mask = flooded_mask & ~initial_mask
    flooded_mask_spine = flooded_mask[zmin:zmax, ymin:ymax, xmin:xmax]
    
    img_coords = np.argwhere(img_spine > 0 & ~flooded_mask_spine)
    img_vals = img_spine[img_spine > 0]

    initial_coords = np.argwhere(initial_mask_spine)
    flooded_coords = np.argwhere(flooded_mask_spine)

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    
    # norm = Normalize(vmin=np.min(img_vals), vmax=np.max(img_vals))
    
    # ax.scatter(img_coords[:, 2], img_coords[:, 1], img_coords[:, 0],
    #            c=plt.cm.gray_r(norm(img_vals)), s=2, alpha=0.3)
    
    ax.scatter(initial_coords[:, 2], initial_coords[:, 1], initial_coords[:, 0],
               c='r', s=30, alpha=0.3, label='initial')
    ax.scatter(flooded_coords[:, 2], flooded_coords[:, 1], flooded_coords[:, 0],
               c='g', s=30, alpha=0.7, label='floodfilled')


    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.view_init(elev=20, azim=135)
    ax.legend()
    plt.tight_layout()
    plt.show()


## Utilities for improved centroid calculation

In [ ]:
from collections import deque
import numpy as np

# Medoid minimizes the sum of distances to the all foreground pixels
def get_medoid(mask):
    coords = np.argwhere(mask)
    dists = np.sum(np.linalg.norm(coords[:, None] - coords[None, :], axis=2), axis=1)
    medoid_index = np.argmin(dists)
    z_medoid, y_medoid, x_medoid = coords[medoid_index]
    return z_medoid, y_medoid, x_medoid

# Weighted median center (instead of weighted mean center, like the centroid) allways gives a
# center in the structure
def get_median(coords, weights):
    sort = np.argsort(coords)
    sorted = coords[sort]
    sorted_weights = weights[sort]
    cum_weights = np.cumsum(sorted_weights)
    total_weight = sorted_weights.sum()
    return sorted[cum_weights >= total_weight / 2][0]
def get_weighted_median_center(mask):
    z_coords, y_coords, x_coords = np.nonzero(mask)
    weights = mask[z_coords, y_coords, x_coords].astype(float)

    x_median = get_median(x_coords, weights)
    y_median = get_median(y_coords, weights)
    z_median = get_median(z_coords, weights)
    return z_median, y_median, x_median


## Improved floodfill

In [ ]:
import matplotlib.pyplot as plt


def floodfill_opt_2(img, initial_mask, dimg, forbidden_mask=None, alpha=0.85, dendrite_ff=False):
    """Flood-fills spine mask labeled with 'label_id' across Z slices from sparse mask annotations,
    using intensity and 3D connectivity. Optionally excludes 'forbidden' voxels.

    Args:
        img (array): 3D image stack (Z, Y, X).
        initial_mask (array): 3D binary mask for current spine/dendrite.
        forbidden_mask (array): 3D binary mask where fill is not allowed (e.g. dendrite).
        alpha (float): parameter that determines the cut on intensity for a voxel to be chosen as seed.
                       Default to 0.85. Should belong to [0, 1].

    Returns:
        array: 3D binary mask after flood-fill.
    """
    # Sanity check
    alpha = max(min(alpha, 1.0), 0.0)
    if forbidden_mask is None:
        forbidden_mask = np.zeros_like(img, dtype=bool)

    # Binary masks to control the visited neighbors and the mask bits added
    visited_mask = np.zeros_like(img, dtype=bool)
    final_mask = np.zeros_like(img, dtype=bool)
    final_mask[initial_mask] = True

    # Spines sometimes have complex shapes, so the centroid may fall in background. Use a center
    # calculated from the weighted medians instead.
    # When floodfilling the dendrite, use max brightness pixel.
    zyx_coords = np.argwhere(initial_mask)
    cz, cy, cx = zyx_coords[np.argmax(img[initial_mask])] if dendrite_ff else get_weighted_median_center(initial_mask)
    # if not dendrite_ff:plt.scatter(cx, cy, c='r', s=10)
    centroid_brightness = img[cz, cy, cx]

    # Initialize queue
    queue = deque()
    queue.extend(get_neighborhood(cz, cy, cx, img.shape))

    # Never allow a too low threshold brightness
    max_z, max_y, max_x = zyx_coords[np.argmax(img[initial_mask])]
    max_brightness = float(img[max_z, max_y, max_x])
    centroid_wrt_max = centroid_brightness / max_brightness
    # Use f(x) = 1-x to only assign higher minimum intensities to centroids with low
    # intensity ratio wrt maximum
    threshold_percentage = 1 - centroid_wrt_max
    threshold = max(centroid_brightness * alpha, 0 if dendrite_ff else max_brightness * threshold_percentage)

    # Use the 99% of the first derivative values (change in intensity) to determine the boundary
    # first derivative cutoff
    img_z_grad = dimg[0] / 2 # gz / 2
    img_z_grad_values = img_z_grad[initial_mask]
    img_z_grad_cutoff = np.percentile(np.abs(img_z_grad_values), 99)

    # Loop over neighbours
    while queue:
        current = queue.popleft()
        z, y, x = current

        # Skip if voxel is already visited or forbidden
        if visited_mask[z, y, x]:
            continue
        else:
            visited_mask[z, y, x] = True
        if forbidden_mask[z, y, x]:
            continue
        # Skip if voxel is too far from the centroid in OZ
        delta_z = cz - z
        if abs(delta_z) > 2 and not dendrite_ff:
            continue
        # Skip if voxel seems close to a boundary
        z_grad = img_z_grad[z, y, x]
        if abs(z_grad) > img_z_grad_cutoff:
            continue

        # If voxel is not bright enough, we might have encountered a boundary
        taylor_1_threshold = threshold - z_grad * delta_z # T_1(x) = f(x0) + f'(x0)(x - x0)
        if img[z, y, x] < taylor_1_threshold:
            continue

        # Add accepted voxel to the mask and its neighbours to the queue
        # if not dendrite_ff:plt.scatter(x, y, c='g', s=10)
        final_mask[z, y, x] = True
        neighbors = get_neighborhood(z, y, x, img.shape)
        for nz, ny, nx in neighbors:
            queue.append((nz, ny, nx))

    return final_mask



In [ ]:
import numpy as np
import imageio as io
from skimage.measure import label
import scipy.ndimage as ndi
import skimage as sk
from scipy import ndimage

def get_dimg(img):
    gz = ndi.prewitt(img, axis=0)
    gy = ndi.prewitt(img, axis=1)
    gx = ndi.prewitt(img, axis=2)
    return gz, gy, gx

img_folder = "test_images"
for animal in {"turtles"}:  # , "mice"}:
    stack = np.asarray(io.mimread(
        f"{img_folder}/out/{animal}_orig.tif"))
    spines = np.asarray(io.mimread(
        f"{img_folder}/out/{animal}_spines.tif"))
    dendrite = np.asarray(io.mimread(
        f"{img_folder}/out/{animal}_dendrite.tif"))

    # Label spines
    L = label(spines)

    # Get all spine labels (exclude background = 0)
    labels = np.unique(L)
    labels = labels[labels != 0]

    # Final mask to accumulate flooded results
    final_mask = np.zeros_like(L, dtype=bool)

    # Optional: store diffs to analyze floodfill additions
    diff_map = np.zeros_like(L, dtype=np.int8)

    # Get image derivative (intensity change rates) in OZ
    denoised = ndi.median_filter(stack, size=3)
    li_thresholded = denoised > sk.filters.threshold_li(denoised)
    dimg = get_dimg(denoised*li_thresholded)

    # Get dendrite mask and extend it over the whole Z axis for avoiding floodfilling spines into dendrite
    dendrite_mask = dendrite == 255
    dendrite_mask_combined = np.any(dendrite_mask, axis=0)
    dendrite_mask_broad = np.broadcast_to(dendrite_mask_combined, dendrite_mask.shape)

    # Floodfill the broadcast mask to account for blurryness of the dendrite
    spines_mask = spines == 255
    spines_mask_combined = np.any(spines_mask, axis=0)
    spines_mask_broad = np.broadcast_to(spines_mask_combined, spines_mask.shape)
    forbidden_mask = spines_mask_broad
    dendrite_mask_ff = floodfill_opt_2(img=stack, initial_mask=dendrite_mask, forbidden_mask=forbidden_mask, dimg=dimg, alpha=0.7, dendrite_ff=True)

    forbidden_mask = dendrite_mask_broad | dendrite_mask_ff

    # Loop through all spines
    for label_id in labels:
        spine_mask = L == label_id
        # print(f"processing label {label_id}")

        # if label_id == 14 or label_id == 19:
        #     plt.figure()
        #     plt.imshow(spine_mask.max(0), cmap='grey')
        #     ys, xs = np.where(spine_mask.max(0))
        #     plt.xlim(xs.min()-100, xs.max()+100)
        #     plt.ylim(ys.max()+100, ys.min()-100)
        # else:
        #   continue

        flooded = floodfill_opt_2(img=stack, initial_mask=spine_mask, forbidden_mask=forbidden_mask, dimg=dimg)

        # plt.show()
        # plot_masked_spines_3d(img=stack, initial_mask=spine_mask, flooded_mask=flooded)

        # Accumulate into final mask
        final_mask |= flooded

        # Optional: difference map
        diff = flooded.astype(int) - spine_mask.astype(int)
        diff_map += diff.astype(np.int8)  # accumulate all diffs

    io.mimwrite(f"{img_folder}/out/{animal}_spines_flooded.tif",
                final_mask.astype(np.uint8) * 255)

    # Save or visualize final mask
    plt.figure()
    plt.imshow(final_mask.max(0), cmap='gray')
    plt.title(f"Final {animal} accumulated mask (max projection)")
    plt.axis('off')
    plt.show()

    # Show total diff (accumulated)
    plt.figure()
    plt.imshow(diff_map.max(0), cmap='bwr', vmin=-1, vmax=1)
    plt.title(f"Total {animal} difference map: red = added, blue = lost")
    plt.axis('off')
    plt.show()



## Create composite

In [ ]:
import cv2
import numpy as np
import tifffile

orig_img = stack
spines_mask = (final_mask > 0).astype(np.uint8) * 255
dendrite_mask = (dendrite_mask > 0).astype(np.uint8) * 255

flooded_overlay = []

for z in range(orig_img.shape[0]):
    s_mask = spines_mask[z]
    d_mask = dendrite_mask[z]
    orig_img_rgb = cv2.cvtColor(orig_img[z].astype(np.uint8), cv2.COLOR_GRAY2BGR)

    color_mask = np.zeros_like(orig_img_rgb)
    color_mask[:, :, 0] = d_mask
    color_mask[:, :, 2] = d_mask
    color_mask[:, :, 1] = s_mask

    overlay = cv2.addWeighted(orig_img_rgb, 1.0, color_mask, 0.5, 0)
    flooded_overlay.append(overlay)

flooded_overlay = np.stack(flooded_overlay, axis=0)

tifffile.imwrite(f"{img_folder}/out/{animal}_spines_flooded_overlay.tif",
    flooded_overlay,
    photometric="rgb"
)
